# Verify preserved prompt-neutral pilot inputs
Attach exact private Kaggle dataset `thestonedape/task-aware-eegtotext`, version 2. Enable Internet and private secret `GITHUB_TOKEN`; CPU is sufficient. This notebook locates the combined pilot-input artifact recursively and revalidates all 215 vector chunks, manifests, indices, trial mappings, prompt masking, and test seals. It does not train or load EEG/model arrays.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '8f44f6cdae02d17e47e4d9ee6fc6b91d16c7076f'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-prompt-neutral-input-verification'
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eegtotext-version-2'
EXPECTED_COMBINED_SHA256 = '6c1fff8d2e89e33a72d03c39651e8ecce678c3b93cdb66747dd6dcc00538cddb'
assert len(COMMIT) == 40 and len(EXPECTED_COMBINED_SHA256) == 64

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE): shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
subprocess.run([sys.executable, '-m', 'unittest', 'evaluation.test_verify_prompt_neutral_pilot_inputs'], check=True, cwd=WORKTREE)
print({'python': platform.python_version(), 'verifier_commit': actual_commit, 'regression': 'PASS'})

In [ ]:
manifest_candidates = glob.glob('/kaggle/input/**/pilot_input_manifest.json', recursive=True)
artifact_roots = []
for path in manifest_candidates:
    root = os.path.dirname(path)
    required = ['eeg/vector_manifest.json', 'eeg/vector_index.csv', 'text/text_vector_manifest.json', 'text/text_vector_index.csv', 'text/trial_text_targets.csv', 'run_metadata.json']
    if all(os.path.isfile(os.path.join(root, item)) for item in required): artifact_roots.append(root)
assert len(artifact_roots) == 1, ('Attach exact dataset version 2 with one complete pilot-input artifact', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''): state.update(block)
    return state.hexdigest()
assert digest(os.path.join(ARTIFACT_ROOT, 'pilot_input_manifest.json')) == EXPECTED_COMBINED_SHA256
print({'preserved_source_id': PRESERVED_SOURCE_ID, 'artifact_root': ARTIFACT_ROOT, 'top_level_files': sorted(os.listdir(ARTIFACT_ROOT))})

In [ ]:
if os.path.exists(OUTPUT): shutil.rmtree(OUTPUT)
os.makedirs(OUTPUT)
report_path = os.path.join(OUTPUT, 'verification_report.json')
subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', 'verify_prompt_neutral_pilot_inputs.py'), '--artifact-root', ARTIFACT_ROOT, '--preserved-source-id', PRESERVED_SOURCE_ID, '--output-report', report_path], check=True)
report = json.load(open(report_path, encoding='utf-8'))
assert report['status'] == 'pass' and report['combined_manifest_sha256'] == EXPECTED_COMBINED_SHA256
assert report['checks'] == {'all_215_chunk_hashes_revalidated': True, 'all_prompt_fields_masked': True, 'all_trial_text_targets_resolve': True, 'held_out_test_accessed': False}
report_sha256 = digest(report_path)
metadata = {'status': 'pass', 'verifier_commit': actual_commit, 'preserved_source_id': PRESERVED_SOURCE_ID, 'combined_manifest_sha256': EXPECTED_COMBINED_SHA256, 'verification_report_sha256': report_sha256, 'test_accessed': False}
with open(os.path.join(OUTPUT, 'verification_run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2, sort_keys=True); handle.write('\n')
summary = {'status': report['status'], 'eeg': report['eeg'], 'text': report['text'], 'combined_manifest_sha256': report['combined_manifest_sha256'], 'verification_report_sha256': report_sha256}
if os.path.exists(WORKTREE): shutil.rmtree(WORKTREE)
print(summary)
print('PROMPT-NEUTRAL PILOT INPUT PRESERVED-ARTIFACT VERIFICATION: PASS')